# Aula 1

## Preparando o ambiente

In [99]:
import duckdb
import pandas as pd
from fuzzywuzzy import fuzz
from fuzzywuzzy.process import extract
"""
pip instal duckdb
pip install pandas
pip install fuzzywuzzy
pip install python-Levenshtein
"""

'\npip instal duckdb\npip install pandas\npip install fuzzywuzzy\npip install python-Levenshtein\n'

In [100]:
# Criando conexão em memória
con = duckdb.connect(database=':memory:')

# Criando schemas
con.execute("CREATE SCHEMA IF NOT EXISTS financeiro")
con.execute("CREATE SCHEMA IF NOT EXISTS operacoes")

# Criando as tabelas com base nos arquivos do chema financeiro
con.execute(f"""
    CREATE TABLE financeiro.clientes AS
    SELECT * FROM read_csv_auto('D:\\Cursos\\ALURA 2026\\GESTAO DE DADOS\\Base de conhecimento\\dados_mestres\\clientes.csv', HEADER=True)
""")
con.execute(f"""
    CREATE TABLE financeiro.pedidos AS
    SELECT * FROM read_csv_auto('D:\\Cursos\\ALURA 2026\\GESTAO DE DADOS\\Base de conhecimento\\dados_mestres\\pedidos.csv', HEADER=True)
""")

# Criando as tabelas com base nos arquivos do schema operações
con.execute(f"""
    CREATE TABLE operacoes.produtos AS
    SELECT * FROM read_csv_auto('D:\\Cursos\\ALURA 2026\\GESTAO DE DADOS\\Base de conhecimento\\dados_mestres\\produtos.csv', HEADER=True)
""")

con.execute(f"""
    CREATE TABLE operacoes.pedidos AS
    SELECT * FROM read_csv_auto('D:\\Cursos\\ALURA 2026\\GESTAO DE DADOS\\Base de conhecimento\\dados_mestres\\pedidos_operacao.csv', HEADER=True)
""")

In [101]:
con.execute('SHOW ALL TABLES').fetchdf()

,database,schema,name,column_names,column_types,temporary
0,memory,financeiro,clientes,"[id_cliente, nome_cliente, endereco, cidade, e...","[BIGINT, VARCHAR, VARCHAR, VARCHAR, VARCHAR, B...",False
1,memory,financeiro,pedidos,"[id_cliente, id_pedido, nome_item, quantidade,...","[BIGINT, BIGINT, VARCHAR, BIGINT, DOUBLE]",False
2,memory,operacoes,pedidos,"[id_produto, quantidade_requisitada, urgencia,...","[BIGINT, BIGINT, VARCHAR, VARCHAR, DATE]",False
3,memory,operacoes,produtos,"[id_produto, descricao_produto, peso_kg, valid...","[BIGINT, VARCHAR, DOUBLE, BIGINT, DOUBLE]",False


## Analisando os dados

In [102]:
clientes = con.execute("select * from financeiro.clientes").df()
clientes

,id_cliente,nome_cliente,endereco,cidade,estado,cnpj
0,1,Queijaria Boa Massa,Rua Boa 282,Belo Horizonte,MG,38704278443537
1,2,Restaurante Sabor da Roça,Rua Sabor 757,Campinas,SP,51118180407845
2,3,Mercado Bom Preço,Rua Bom 279,Curitiba,PR,23579980402940
3,4,Laticínios do Sul,Rua do 775,Porto Alegre,RS,95842293722549
4,5,Supermercado Leite e Mel,Rua Leite 676,Recife,PE,29073506471229
5,6,Cantina da Serra,Rua da 38,Fortaleza,CE,42033300135209
6,7,Empório do Queijo Canastra,Rua do 229,Salvador,BA,95546201460520
7,8,Restaurante Tacho de Ouro,Rua Tacho 923,Florianópolis,SC,30142799259280
8,9,Mercado Sítio Feliz,Rua Sítio 403,Goiânia,GO,84323816906331
9,10,Laticínios Vale Verde,Rua Vale 177,Brasília,DF,88729707173011


## Identificando o problema

* Dados Faltantes (Cidade)
* Dados Duplicados (ID)
* Dados Inválidos (CNPJ)
* Dados Conflitantes


## Dados Faltantes

In [103]:
clientes.isna().sum()

id_cliente      0
nome_cliente    0
endereco        0
cidade          1
estado          0
cnpj            0
dtype: int64

## Dados Duplicados

In [104]:
contagem = clientes.estado.value_counts()
duplicados = contagem[contagem > 1]

if duplicados.empty:
    print("Não há estados duplicados na coluna 'estado'.")
else:
    print("Estados duplicados na coluna 'estado':")
    print(duplicados)

Estados duplicados na coluna 'estado':
estado
MG    2
SP    2
BA    2
AM    2
Name: count, dtype: int64


In [105]:
clientes[clientes.cidade == "Manaus"]

,id_cliente,nome_cliente,endereco,cidade,estado,cnpj
18,19,Restaurante Fazenda & Sabor,Rua Fazenda 531,Manaus,AM,80042646533900
24,24,Hotel Fazenda & Sabor,Rua Fazenda 531,Manaus,AM,80042


In [106]:
clientes[clientes.estado == "MG"]

,id_cliente,nome_cliente,endereco,cidade,estado,cnpj
0,1,Queijaria Boa Massa,Rua Boa 282,Belo Horizonte,MG,38704278443537
10,11,Bodega do Sabor,Rua do 497,Uberlândia,MG,45318018583501


In [107]:
clientes[clientes.estado == "BA"]

,id_cliente,nome_cliente,endereco,cidade,estado,cnpj
6,7,Empório do Queijo Canastra,Rua do 229,Salvador,BA,95546201460520
22,22,Emp. do Queijo Canastra,Rua do 229,Salvador,BA,9554620


In [108]:
clientes[clientes.estado == "SP"]

,id_cliente,nome_cliente,endereco,cidade,estado,cnpj
1,2,Restaurante Sabor da Roça,Rua Sabor 757,Campinas,SP,51118180407845
23,23,Rest. Savor da Roca,Rua Sabor 757,Campinas,SP,51118180407845


# Correções a serem feitas

* Dados Faltantes (Cidade)
* Dados Duplicados (ID)
* Dados Inválidos (CNPJ)
* Dados Conflitantes

# Definindo Plano de Ação

1. Criação das regras de qualidade de dados
2. Retificar os Dados que não passam na qualidade de dados
3. Identificar a responsabilidade sobre os dados

# 1 - Criação de qualidade de dados

1. Consistência - Os mesmos dados devem ter os mesmos valores em diferentes tabelas, sistemas ou momentos, sem contradições.
2. Conformidade - Os dados seguem um formato, padrão ou regra de negócio definido (máscaras, domínios, unidades, padrões do setor).
3. Disponibilidade - O dado está acessível quando e onde é necessário — sem indisponibilidade, latência excessiva ou falta de permissão.
4. Integridade - Os dados mantêm relacionamentos e restrições corretos (chaves primárias, estrangeiras, unicidade, não-nulos)
5. Precisão - O dado representa corretamente a realidade que ele descreve.
6. Completude - Todos os dados necessários estão presentes — sem campos faltantes, nulos indevidos ou registros ausentes.


<img src="fluxo_negocio_empresa_laticinios.png" alt="SCHEMAS" width="2000" height="1000">

Será que o estado do registro é válido ? -> testar se o estado é um dos estados brasileiros

Será que existe a cidade informada realmente pertence aquele estado ? -> testar se aquela cidade pertence aquele estado.

Será que este registro está completo ? -> testar se existe algum dado nulo.

Será que o CNPJ atende à conformidade ? -> Testar se o CNPJ tem 14 digitos

Será que existe mais de um registro para o mesmo ID ?

Será que existe mais de um registro para um determinado cliente ?

# Dados de Referência
Características principais
*  **Estabilidade** - Mudam raramente.
* **Compartilhamento** - São usados por múltiplas áreas, sistemas e processos.
* **Padronização** - Seguem normas externas ou internas.
* **Volume pequeno** - São conjuntos pequenos de dados.
* **Baixa complexidade estrutural** -
Geralmente tabelas simples
* **Alta criticidade** -
Não pode ter erros em dados de referência porque propagam-se por todo o ecossistema.
* **Governança forte** -
Devem ter dono (owner), curador e processo formal de mudança.
* **Origem geralmente externa**

In [109]:
dados_municipios = pd.read_excel("D:\\Cursos\\ALURA 2026\\GESTAO DE DADOS\\Base de conhecimento\\dados_mestres\\RELATORIO_DTB_BRASIL_MUNICIPIO.xls",skiprows=6)

In [110]:
dados_municipios[dados_municipios.Nome_Município == "Campo Grande"]

,UF,Nome_UF,Região Geográfica Intermediária,Nome Região Geográfica Intermediária,Região Geográfica Imediata,Nome Região Geográfica Imediata,Município,Código Município Completo,Nome_Município
1097,24,Rio Grande do Norte,2403,Mossoró,240009,Mossoró,1305,2401305,Campo Grande
1665,27,Alagoas,2702,Arapiraca,270007,Arapiraca,1506,2701506,Campo Grande
5122,50,Mato Grosso do Sul,5001,Campo Grande,500001,Campo Grande,2704,5002704,Campo Grande


In [111]:
dados_municipios = dados_municipios[["Nome_UF", "Nome_Município"]]
dados_municipios.columns = ["nome_uf", "nome_cidade"]

In [112]:
dados_municipios

,nome_uf,nome_cidade
0,Rondônia,Alta Floresta D'Oeste
1,Rondônia,Alto Alegre dos Parecis
2,Rondônia,Alto Paraíso
3,Rondônia,Alvorada D'Oeste
4,Rondônia,Ariquemes
...,...,...
5565,Goiás,Vianópolis
5566,Goiás,Vicentinópolis
5567,Goiás,Vila Boa
5568,Goiás,Vila Propício


### Solicitado ao DeepSeek criar o Dicionário a seguir para adicionar a sigla Sigla referente ao nome do Estado

In [113]:
mapa_siglas = {
    'Acre': 'AC',
    'Alagoas': 'AL',
    'Amapá': 'AP',
    'Amazonas': 'AM',
    'Bahia': 'BA',
    'Ceará': 'CE',
    'Distrito Federal': 'DF',
    'Espírito Santo': 'ES',
    'Goiás': 'GO',
    'Maranhão': 'MA',
    'Mato Grosso': 'MT',
    'Mato Grosso do Sul': 'MS',
    'Minas Gerais': 'MG',
    'Pará': 'PA',
    'Paraíba': 'PB',
    'Paraná': 'PR',
    'Pernambuco': 'PE',
    'Piauí': 'PI',
    'Rio de Janeiro': 'RJ',
    'Rio Grande do Norte': 'RN',
    'Rio Grande do Sul': 'RS',
    'Rondônia': 'RO',
    'Roraima': 'RR',
    'Santa Catarina': 'SC',
    'São Paulo': 'SP',
    'Sergipe': 'SE',
    'Tocantins': 'TO',
}



In [114]:
dados_municipios['sigla_uf'] = dados_municipios.nome_uf.map(mapa_siglas)

In [115]:
dados_municipios

,nome_uf,nome_cidade,sigla_uf
0,Rondônia,Alta Floresta D'Oeste,RO
1,Rondônia,Alto Alegre dos Parecis,RO
2,Rondônia,Alto Paraíso,RO
3,Rondônia,Alvorada D'Oeste,RO
4,Rondônia,Ariquemes,RO
...,...,...,...
5565,Goiás,Vianópolis,GO
5566,Goiás,Vicentinópolis,GO
5567,Goiás,Vila Boa,GO
5568,Goiás,Vila Propício,GO


In [116]:
con.execute("CREATE SCHEMA IF NOT EXISTS dados")
con.register("view_municípios", dados_municipios)
con.execute("CREATE OR REPLACE VIEW dados.municípios AS SELECT * FROM view_municípios")


In [117]:
con.execute("select * from dados.municípios").df()

,nome_uf,nome_cidade,sigla_uf
0,Rondônia,Alta Floresta D'Oeste,RO
1,Rondônia,Alto Alegre dos Parecis,RO
2,Rondônia,Alto Paraíso,RO
3,Rondônia,Alvorada D'Oeste,RO
4,Rondônia,Ariquemes,RO
...,...,...,...
5565,Goiás,Vianópolis,GO
5566,Goiás,Vicentinópolis,GO
5567,Goiás,Vila Boa,GO
5568,Goiás,Vila Propício,GO


# Biblioteca fuzzywuzzy

In [118]:
fuzz.ratio("pato","gato")

75

In [119]:
clientes[clientes.estado == "SP"]

,id_cliente,nome_cliente,endereco,cidade,estado,cnpj
1,2,Restaurante Sabor da Roça,Rua Sabor 757,Campinas,SP,51118180407845
23,23,Rest. Savor da Roca,Rua Sabor 757,Campinas,SP,51118180407845


In [120]:
fuzz.ratio("Restaurante Sabor da Roça","Rest. Savor da Roca")

73

In [121]:
clientes[clientes.estado == "AM"]

,id_cliente,nome_cliente,endereco,cidade,estado,cnpj
18,19,Restaurante Fazenda & Sabor,Rua Fazenda 531,Manaus,AM,80042646533900
24,24,Hotel Fazenda & Sabor,Rua Fazenda 531,Manaus,AM,80042


In [122]:
fuzz.ratio("Restaurante Fazenda & Sabor","Hotel Fazenda & Sabor")

75

# Aplicando o fuzzywuzzy para o nosso negócio

In [123]:
possibilidades = ["pato","gato","vaca","paca"]

In [124]:
extract("patos",possibilidades, limit=3)

[('pato', 89), ('gato', 67), ('paca', 44)]

In [125]:
def compara_candidatos_por_cidade(df: pd.DataFrame,
                                  cidade: str,
                                  nivel_similaridade: int = 70) -> list:
    """
    Compara nomes de clientes na mesma cidade e retorna pares com
    similaridade >= nivel_similaridade.
    """
    duplicados = []  # ← acumulador que será retornado

    df_cidade = df[df.cidade == cidade]
    candidatos = df_cidade.nome_cliente.to_list()

    if len(candidatos) < 2:
        return duplicados  # ← retorno garantido mesmo sem pares

    # Compara cada par apenas UMA vez (i < j)
    for i, candidato in enumerate(candidatos):
        for outro in candidatos[i+1:]:
            similaridade = fuzz.ratio(candidato, outro)
            if similaridade >= nivel_similaridade:
                print(f"'{candidato}' possui similaridade de {similaridade}% com \n'{outro}'")
                duplicados.append({
                    'cidade': cidade,
                    'cliente_1': candidato,
                    'cliente_2': outro,
                    'similaridade': similaridade,
                })
            else:
                print("---")

    return duplicados  # ← ESSENCIAL: retorna a lista

In [126]:
compara_candidatos_por_cidade(clientes,"Manaus")

'Restaurante Fazenda & Sabor' possui similaridade de 75% com 
'Hotel Fazenda & Sabor'


[{'cidade': 'Manaus',
  'cliente_1': 'Restaurante Fazenda & Sabor',
  'cliente_2': 'Hotel Fazenda & Sabor',
  'similaridade': 75}]

# Melhorando o Código

In [127]:
lista_de_cidades_com_mais_de_um_cliente = clientes[clientes.cidade.duplicated()]
lista_de_cidades_com_mais_de_um_cliente = lista_de_cidades_com_mais_de_um_cliente.cidade.unique()


In [128]:
lista_de_cidades_com_mais_de_um_cliente

<StringArray>
['Salvador', 'Campinas', 'Manaus']
Length: 3, dtype: str

In [129]:
todos_os_duplicados = []
for cidade in lista_de_cidades_com_mais_de_um_cliente:
  todos_os_duplicados += compara_candidatos_por_cidade(clientes, cidade)

'Empório do Queijo Canastra' possui similaridade de 90% com 
'Emp. do Queijo Canastra'
'Restaurante Sabor da Roça' possui similaridade de 73% com 
'Rest. Savor da Roca'
'Restaurante Fazenda & Sabor' possui similaridade de 75% com 
'Hotel Fazenda & Sabor'


In [130]:
todos_os_duplicados

[{'cidade': 'Salvador',
  'cliente_1': 'Empório do Queijo Canastra',
  'cliente_2': 'Emp. do Queijo Canastra',
  'similaridade': 90},
 {'cidade': 'Campinas',
  'cliente_1': 'Restaurante Sabor da Roça',
  'cliente_2': 'Rest. Savor da Roca',
  'similaridade': 73},
 {'cidade': 'Manaus',
  'cliente_1': 'Restaurante Fazenda & Sabor',
  'cliente_2': 'Hotel Fazenda & Sabor',
  'similaridade': 75}]

# Criando Nova Base de Clientes

In [131]:
novos_clientes = clientes.copy()

In [132]:
novos_clientes.loc[clientes.id_cliente == 18, "cidade"] = "Campo Grande"

In [133]:
novos_clientes

,id_cliente,nome_cliente,endereco,cidade,estado,cnpj
0,1,Queijaria Boa Massa,Rua Boa 282,Belo Horizonte,MG,38704278443537
1,2,Restaurante Sabor da Roça,Rua Sabor 757,Campinas,SP,51118180407845
2,3,Mercado Bom Preço,Rua Bom 279,Curitiba,PR,23579980402940
3,4,Laticínios do Sul,Rua do 775,Porto Alegre,RS,95842293722549
4,5,Supermercado Leite e Mel,Rua Leite 676,Recife,PE,29073506471229
5,6,Cantina da Serra,Rua da 38,Fortaleza,CE,42033300135209
6,7,Empório do Queijo Canastra,Rua do 229,Salvador,BA,95546201460520
7,8,Restaurante Tacho de Ouro,Rua Tacho 923,Florianópolis,SC,30142799259280
8,9,Mercado Sítio Feliz,Rua Sítio 403,Goiânia,GO,84323816906331
9,10,Laticínios Vale Verde,Rua Vale 177,Brasília,DF,88729707173011


# Refinando os Dados

In [134]:
todos_os_duplicados

[{'cidade': 'Salvador',
  'cliente_1': 'Empório do Queijo Canastra',
  'cliente_2': 'Emp. do Queijo Canastra',
  'similaridade': 90},
 {'cidade': 'Campinas',
  'cliente_1': 'Restaurante Sabor da Roça',
  'cliente_2': 'Rest. Savor da Roca',
  'similaridade': 73},
 {'cidade': 'Manaus',
  'cliente_1': 'Restaurante Fazenda & Sabor',
  'cliente_2': 'Hotel Fazenda & Sabor',
  'similaridade': 75}]

* Correto: Empório do Queijo Canastra (ID 7) Errado (ID 22)
* Correto: Restaurante Sabor da Roça (ID 2) Errado (ID 23)
* Correto: Restaurante Fazenda & Sabor (ID 19) ERRADO (ID 24)

In [135]:
novos_clientes = novos_clientes.drop(22)
novos_clientes = novos_clientes.drop(23)
novos_clientes = novos_clientes.drop(24)

In [136]:
novos_clientes.head(5)

,id_cliente,nome_cliente,endereco,cidade,estado,cnpj
0,1,Queijaria Boa Massa,Rua Boa 282,Belo Horizonte,MG,38704278443537
1,2,Restaurante Sabor da Roça,Rua Sabor 757,Campinas,SP,51118180407845
2,3,Mercado Bom Preço,Rua Bom 279,Curitiba,PR,23579980402940
3,4,Laticínios do Sul,Rua do 775,Porto Alegre,RS,95842293722549
4,5,Supermercado Leite e Mel,Rua Leite 676,Recife,PE,29073506471229


In [137]:
novos_clientes[clientes.id_cliente == 21]

C:\Users\Nino\AppData\Local\Temp\ipykernel_11304\4126189750.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  novos_clientes[clientes.id_cliente == 21]


,id_cliente,nome_cliente,endereco,cidade,estado,cnpj
20,21,Mercado Silva,Rua de Sá 121,Vitória,ES,43667789271700
21,21,Mercado Silva,Rua João de Barro 2,Manaus,AM,43668953170021


# Para resolver registros com 2 registros vamos primeiro alterar o id do Mercado Silva que está em Manaus

In [138]:
novos_clientes.loc[novos_clientes.cnpj == 43668953170021 , "id_cliente"] = 30

In [139]:
novos_clientes

,id_cliente,nome_cliente,endereco,cidade,estado,cnpj
0,1,Queijaria Boa Massa,Rua Boa 282,Belo Horizonte,MG,38704278443537
1,2,Restaurante Sabor da Roça,Rua Sabor 757,Campinas,SP,51118180407845
2,3,Mercado Bom Preço,Rua Bom 279,Curitiba,PR,23579980402940
3,4,Laticínios do Sul,Rua do 775,Porto Alegre,RS,95842293722549
4,5,Supermercado Leite e Mel,Rua Leite 676,Recife,PE,29073506471229
5,6,Cantina da Serra,Rua da 38,Fortaleza,CE,42033300135209
6,7,Empório do Queijo Canastra,Rua do 229,Salvador,BA,95546201460520
7,8,Restaurante Tacho de Ouro,Rua Tacho 923,Florianópolis,SC,30142799259280
8,9,Mercado Sítio Feliz,Rua Sítio 403,Goiânia,GO,84323816906331
9,10,Laticínios Vale Verde,Rua Vale 177,Brasília,DF,88729707173011


Será que o estado do registro é válido ? -> testar se o estado é um dos estados brasileiros

OK - Será que existe a cidade informada realmente pertence aquele estado ? -> testar se aquela cidade pertence aquele estado.

OK - Será que este registro está completo ? -> testar se existe algum dado nulo.

OK - Será que o CNPJ atende à conformidade ? -> Testar se o CNPJ tem 14 digitos

OK - Será que existe mais de um registro para o mesmo ID ?

OK - Será que existe mais de um registro para um determinado cliente ?

In [140]:
def valida_dados_nulos(df: pd.DataFrame)-> bool:
  if df.isna().sum().sum() > 0:
    return False
  else:
    return True


In [141]:
valida_dados_nulos(novos_clientes)

True

In [142]:
def valida_sem_duplicidade_ids(df: pd.DataFrame)-> bool:
  resultado = df.id_cliente.duplicated().sum()
  if resultado > 0:
    return False
  else:
    return True

In [143]:
valida_sem_duplicidade_ids(novos_clientes)

True

# Continuando a validação

In [144]:
def valida_estado(df: pd.DataFrame):
  todos_os_estados = set(dados_municipios.sigla_uf)
  estados_contidos = set(df.estado)
  print(estados_contidos - todos_os_estados)

# Verifica-se que existem 2 registros não válidos. ' AM' e ' ES' que tem um caracter de espaço em branco na cigla do Estado.

In [145]:
valida_estado(novos_clientes)

{' ES', ' AM'}


# Corrigindo fazemos um STRIP na coluna

In [146]:
novos_clientes.estado = novos_clientes.estado.str.strip()

In [147]:
def valida_cidade(df: pd.DataFrame):
  for i, row in df.iterrows():
    if row.cidade not in dados_municipios[dados_municipios.sigla_uf == row.estado].nome_cidade.unique():
      print(row)
      return False
  return True


## Verificamos que ' Vitória' não pertence ao Espírito Santo porque está com um espaço em branco a mais. Para resolver isso vamos dar Strip na coluna cidade.

In [148]:
valida_cidade(novos_clientes)

id_cliente                  21
nome_cliente     Mercado Silva
endereco         Rua de Sá 121
cidade                 Vitória
estado                      ES
cnpj            43667789271700
Name: 20, dtype: object


False

# Fazendo Strip em coluna Estado e Cidade

In [149]:
novos_clientes.estado = novos_clientes.estado.str.strip()
novos_clientes.cidade = novos_clientes.cidade.str.strip()

In [150]:
valida_cidade(novos_clientes)

True

# Validação CNPJ

In [151]:
def valida_cnpj(df: pd.DataFrame):
    return bool(df.cnpj.apply(lambda x: len(str(x)) == 14).all())

In [152]:
def retorna_cnpj_invalido(df: pd.DataFrame):
    return df[df.cnpj.apply(lambda x: len(str(x)) != 14)]

In [153]:
if valida_cnpj(novos_clientes) == False:
  retorna_cnpj_invalido(novos_clientes)
else:
  print("Todos os CNPJ's estão válidos")

In [154]:
retorna_cnpj_invalido(novos_clientes)

,id_cliente,nome_cliente,endereco,cidade,estado,cnpj
19,20,Empório da Canastra,Rua da 112,Macapá,AP,4346975957570


In [155]:
len(str(novos_clientes.iloc[19].cnpj))

13

# Teóricamente a área de negócio definiu que para corrigir este cnpj deve adicionar um zero ao final do cnpj

In [156]:
novos_clientes.loc[novos_clientes.cnpj == 4346975957570, "cnpj"] = 43469759575700

In [157]:
if valida_cnpj(novos_clientes) == False:
  retorna_cnpj_invalido(novos_clientes)
else:
  print("Todos os CNPJ's estão válidos")

Todos os CNPJ's estão válidos


# Salvando a nova base de dados de clientes

In [158]:
con.register("view_cliente_corrigida", novos_clientes)

In [159]:
con.execute("CREATE TABLE dados.clientes_validos AS SELECT * FROM view_cliente_corrigida")

In [160]:
con.execute("SHOW ALL TABLES").fetchdf()

,database,schema,name,column_names,column_types,temporary
0,memory,dados,clientes_validos,"[id_cliente, nome_cliente, endereco, cidade, e...","[BIGINT, VARCHAR, VARCHAR, VARCHAR, VARCHAR, B...",False
1,memory,dados,municípios,"[nome_uf, nome_cidade, sigla_uf]","[VARCHAR, VARCHAR, VARCHAR]",False
2,memory,financeiro,clientes,"[id_cliente, nome_cliente, endereco, cidade, e...","[BIGINT, VARCHAR, VARCHAR, VARCHAR, VARCHAR, B...",False
3,memory,financeiro,pedidos,"[id_cliente, id_pedido, nome_item, quantidade,...","[BIGINT, BIGINT, VARCHAR, BIGINT, DOUBLE]",False
4,memory,operacoes,pedidos,"[id_produto, quantidade_requisitada, urgencia,...","[BIGINT, BIGINT, VARCHAR, VARCHAR, DATE]",False
5,memory,operacoes,produtos,"[id_produto, descricao_produto, peso_kg, valid...","[BIGINT, VARCHAR, DOUBLE, BIGINT, DOUBLE]",False
6,temp,main,view_cliente_corrigida,"[id_cliente, nome_cliente, endereco, cidade, e...","[BIGINT, VARCHAR, VARCHAR, VARCHAR, VARCHAR, B...",True
7,temp,main,view_municípios,"[nome_uf, nome_cidade, sigla_uf]","[VARCHAR, VARCHAR, VARCHAR]",True
